# Booklet 1: Disambiguation

In this first booklet, we’ll go through how the disambiguation model works. Remembering that our goal is to resolve ambiguous morphological readings by the FST, we will identify the ambiguity and see how the CG3 rules help us resolve it.



### Running the disambiguation

We’ll work with the following sentence, which can be found in the "Sentence Examples" section of ([this OPD entry](https://ojibwe.lib.umn.edu/main-entry/mitig-na)):

**Gidaa-ozhiga'waa na awedi mitig.**

Translation: Can you tap that tree over there?

To be explicit about what the model sees before disambiguation and what it returns after, we’ll call `disambiguate()` with `verbose=True`. This prints both the CG3 formatted input and the CG3 output after pruning, so you don’t need separate calls to preview and then run. 


In [9]:
# this block can be ignored, just setting up the path
import sys
import os
sys.path.append(os.path.abspath("../.."))  

In [10]:
from fst_runtime.fst import Fst
from src.disambiguation import disambiguate

# getting the FST (make sure path is correct)
fst = Fst("../../data/fst/ojibwe.att")
# path to the grammar (again, make sure path is correct)
grammar = "../../data/rules/disambiguation.cg3"
# sentence to be analyzed
sent = "Gidaa-ozhiga'waa na awedi mitig."

# calling disambiguate() with verbose set to True
out = disambiguate(sent, grammar, fst, verbose=True)

Before parsing:
"<gidaa-ozhiga'waa>"
	"ozhiga'wi" PVTense/daa VTA Ind Pos Neu 2SgSubj 3SgProxObj
	"ozhiga'" PVTense/daa VTA Ind Pos Neu 2SgSubj 3SgProxObj
"<na>"
	"na" PCDisc
"<awedi>"
	"awedi" PRONDem NA ProxSg
"<mitig>"
	"mitig" NA ProxSg
	"mitig" NI Sg
"<.>"


--------------------
After parsing:
"<gidaa-ozhiga'waa>"
	"ozhiga'" PVTense/daa VTA Ind Pos Neu 2SgSubj 3SgProxObj
"<na>"
	"na" PCDisc
"<awedi>"
	"awedi" PRONDem NA ProxSg
"<mitig>"
	"mitig" NA ProxSg
"<.>"




**Before parsing**:

Looking at the output, we see that there are two ambiguities: 
- *mitig*, which has two readings (`NA ProxSg` vs `NI Sg`).
- *gidaa-ozhiga'waa*, which only differs in its two lemmas (`"ozhiga'wi"` vs. `"ozhiga'"`)

**After parsing:**

Here we see that *mitig* lost the inanimate noun reading (`NI Sg`) and kept the animate reading (`NA ProxSg`), because it was immediately preceded by *awedi*, an unambiguous animate demonstrative pronoun. 
The other ambiguity concerning two lemmas， `"ozhiga'wi"` vs. `"ozhiga'"`, is an example of a FST-produced ambiguity that is removed across the board, and not based on context. Here the simple explanation is that the OPD (on which the FST was built) listed two different lemmas for the dictionary entry for this given word, and in these cases we choose the more common analysis, which in this case is `"ozhiga'"`. 

### Using the CG3 IDE for development

In the above output, we don't actually see which of the disambiguation rules fired to disambiguate the reading. This information is shown when running inputs in the [CG3 IDE](https://edu.visl.dk/cg3ide.html), which is the best tool to use when actually writing / debugging the CG3 file. 

If you are using the CG3 IDE, the `ojibwe_sentence_to_cg3_format` function can be used to parse the Ojibwe text into CG3 format without running disambiguation. This is useful because it can be copy pasted directly into the input section of the IDE to debug from there. Here is an example run:

In [11]:
from src.disambiguation import ojibwe_sentence_to_cg3_format
cg3_input = ojibwe_sentence_to_cg3_format(sent, fst)
print(cg3_input)

"<gidaa-ozhiga'waa>"
	"ozhiga'wi" PVTense/daa VTA Ind Pos Neu 2SgSubj 3SgProxObj
	"ozhiga'" PVTense/daa VTA Ind Pos Neu 2SgSubj 3SgProxObj
"<na>"
	"na" PCDisc
"<awedi>"
	"awedi" PRONDem NA ProxSg
"<mitig>"
	"mitig" NA ProxSg
	"mitig" NI Sg
"<.>"




### Summary of the `disambiguation.py` module

To use this module, you generally only need to call the `disambiguate()` function on the Ojibwe text to get the expected CG3 outputs. If you are using the CG3 IDE, you can also use `ojibwe_sentence_to_cg3_format()` to simply get CG3 formatted text with the morphological readings.

That's everything for the disambiguation module!